# Train world model (M3 — DreamerV3 reset)

**If the training cell contains `recon_embed`, this file is a stale editor buffer.
Do not save. Reload from disk, then Kernel → Restart → Run All.**

Live graph: `encoder → RSSM.observe → feat=concat(h, z_posterior) → one decoder`.
Config: `configs/m3_dreamer_s.yaml`. `RESUME = None`. Old 12k / 131M checkpoints
will not load. Missing `data/m3_dreamer_s_replay.pt` → next cell collects 600
random Crafter episodes, then 50k train steps.


In [1]:
from __future__ import annotations

import importlib
import json
import os
import random
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "src").is_dir():
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            ROOT = candidate
            break
os.chdir(ROOT)
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import clear_output, display
from torch.utils.tensorboard import SummaryWriter

import models.decoder as decoder_mod
import models.encoder as encoder_mod
import models.heads as heads_mod
import models.preprocess as preprocess_mod
import models.rssm as rssm_mod
import models.symlog as symlog_mod
import models.world_model as wm_mod
import training.device as device_mod
import training.losses as losses_mod
import training.replay_buffer as replay_mod
import training.wm_step as wm_step_mod

# Reload modules before binding names so a stale kernel cannot keep old APIs.
# (another notebook, or an old cell) cannot keep stale names.
importlib.reload(preprocess_mod)
importlib.reload(symlog_mod)
importlib.reload(heads_mod)
importlib.reload(encoder_mod)
importlib.reload(decoder_mod)
importlib.reload(rssm_mod)
importlib.reload(wm_mod)
importlib.reload(losses_mod)
importlib.reload(device_mod)
importlib.reload(replay_mod)
importlib.reload(wm_step_mod)

from models.preprocess import nchw_unit_to_nhwc_uint8, nhwc_uint8_to_nchw_unit
from models.symlog import symlog_twohot_mean
from models.world_model import WorldModel
from training.device import (
    autocast_context,
    configure_runtime,
    describe_device,
    get_device,
    make_grad_scaler,
    parse_amp,
    to_device,
    vram_peak_gb,
    warn_if_not_cuda,
)
from training.replay_buffer import ReplayBuffer, collect_random_episodes
from training.wm_step import world_model_step

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass


CONFIG = Path("configs/m3_dreamer_s.yaml")
RESUME = None
STEPS_OVERRIDE: int | None = None
RECOLLECT = False

with CONFIG.open() as f:
    cfg = yaml.safe_load(f)

seed = int(cfg["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = get_device()
configure_runtime(device)
if device.type == "cuda":
    torch.cuda.manual_seed_all(seed)
    torch.cuda.reset_peak_memory_stats()
print(f"device: {describe_device(device)}")
warn_if_not_cuda(device)

t_cfg = cfg["train"]
print(
    f"amp={t_cfg.get('amp', 'bf16')} batch={t_cfg['batch_size']} seq_len={t_cfg['seq_len']} "
    f"lr={t_cfg['lr']} recon_scale={t_cfg['recon_scale']} reward_scale={t_cfg['reward_scale']} "
    f"continue_scale={t_cfg['continue_scale']} kl_scale={t_cfg['kl_scale']}"
)
print(
    f"dyn_scale={t_cfg['dyn_scale']} rep_scale={t_cfg['rep_scale']} "
    f"free_nats={t_cfg['free_nats']} free_nats_dyn={t_cfg.get('free_nats_dyn')} "
    f"deter={cfg['rssm']['deter_dim']} hidden={cfg['rssm']['hidden']}"
)


device: cuda  NVIDIA GeForce RTX 5080  15.9 GiB  cc12.0  torch 2.11.0+cu128
amp=bf16 batch=16 seq_len=32 lr=0.0001 recon_scale=1.0 reward_scale=1.0 continue_scale=1.0 kl_scale=1.0
dyn_scale=0.5 rep_scale=0.1 free_nats=1.0 free_nats_dyn=None deter=512 hidden=512


In [2]:
replay_path = Path(cfg["collect"]["out_path"])
if RECOLLECT or not replay_path.exists():
    n_ep = int(cfg["collect"]["num_episodes"])
    print(
        f"collecting {n_ep} episodes from {cfg['env']['id']} "
        f"(max {cfg['collect']['max_episode_steps']} steps) — live per-episode log below",
        flush=True,
    )
    buf = collect_random_episodes(
        env_id=str(cfg["env"]["id"]),
        num_episodes=n_ep,
        max_episode_steps=int(cfg["collect"]["max_episode_steps"]),
        action_dim=int(cfg["env"]["action_dim"]),
        seed=seed,
        progress=True,
    )
    replay_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(buf.state_dict(), replay_path)
    print(f"wrote {replay_path}: episodes={len(buf)} steps={buf.num_steps}", flush=True)
else:
    print(f"using existing {replay_path}", flush=True)

buffer = ReplayBuffer(seed=seed)
buffer.load_state_dict(torch.load(replay_path, weights_only=False))
print(f"replay: episodes={len(buffer)} steps={buffer.num_steps}", flush=True)


collecting 600 episodes from CrafterReward-v1 (max 400 steps) — live per-episode log below
collecting 600 random episodes from CrafterReward-v1 (max 400 steps/ep, seed=0)
  [   1/600]   0.2%  len=161  ret= +2.10  steps=161      1.46 ep/s  elapsed 1s  eta 6m51s
  [   2/600]   0.3%  len=245  ret= +3.10  steps=406      1.39 ep/s  elapsed 1s  eta 7m09s
  [   3/600]   0.5%  len=223  ret= +1.10  steps=629      1.42 ep/s  elapsed 2s  eta 7m01s
  [   4/600]   0.7%  len=154  ret= +0.10  steps=783      1.44 ep/s  elapsed 3s  eta 6m53s
  [   5/600]   0.8%  len=174  ret= +1.10  steps=957      1.45 ep/s  elapsed 3s  eta 6m51s
  [   6/600]   1.0%  len=175  ret= +0.10  steps=1132     1.46 ep/s  elapsed 4s  eta 6m48s
  [   7/600]   1.2%  len=160  ret= +1.10  steps=1292     1.47 ep/s  elapsed 5s  eta 6m45s
  [   8/600]   1.3%  len=154  ret= +0.10  steps=1446     1.47 ep/s  elapsed 5s  eta 6m42s
  [   9/600]   1.5%  len=292  ret= +2.10  steps=1738     1.45 ep/s  elapsed 6s  eta 6m48s
  [  10/600]   1.7%

In [3]:
enc, rssm_cfg, dec, heads, train_cfg = (
    cfg["encoder"],
    cfg["rssm"],
    cfg.get("decoder", {}),
    cfg.get("heads", {}),
    cfg["train"],
)
model = WorldModel.from_config_dims(
    embed_dim=int(enc["embed_dim"]),
    encoder_channels=tuple(int(c) for c in enc["channels"]),
    action_dim=int(cfg["env"]["action_dim"]),
    deter_dim=int(rssm_cfg["deter_dim"]),
    stoch=int(rssm_cfg["stoch"]),
    classes=int(rssm_cfg["classes"]),
    hidden=int(rssm_cfg["hidden"]),
    unimix=float(rssm_cfg.get("unimix", 0.01)),
    act=str(rssm_cfg.get("act", "silu")),
    initial=str(rssm_cfg.get("initial", "learned")),
    rec_depth=int(rssm_cfg.get("rec_depth", 1)),
    prior_layers=int(rssm_cfg.get("prior_layers", 1)),
    decoder_channels=tuple(int(c) for c in dec.get("channels", [256, 128, 64, 32])),
    head_hidden=int(heads.get("hidden", 512)),
    head_layers=int(heads.get("layers", 2)),
    encoder_blocks=int(enc.get("blocks", 1)),
    decoder_blocks=int(dec.get("blocks", 1)),
    output_activation=str(dec.get("output_activation", "linear")),
    reward_num_bins=int(heads.get("reward_bins", 255)),
    reward_low=float(heads.get("reward_low", -20.0)),
    reward_high=float(heads.get("reward_high", 20.0)),
).to(device)
optim = torch.optim.Adam(model.parameters(), lr=float(train_cfg["lr"]))
start_step = 0
if RESUME is not None:
    ckpt = torch.load(Path(RESUME), weights_only=False, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    if "optim" in ckpt:
        optim.load_state_dict(ckpt["optim"])
    start_step = int(ckpt.get("step", 0))
    print(f"resumed from {RESUME} at step {start_step}")
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
amp_dtype = parse_amp(train_cfg.get("amp", "bf16"), device)
scaler = make_grad_scaler(device, amp_dtype)
print(
    f"amp: {train_cfg.get('amp', 'bf16')}  batch={train_cfg['batch_size']}  "
    f"seq_len={train_cfg['seq_len']}"
)


params: 18.72M
amp: bf16  batch=16  seq_len=32


In [4]:
# ONE decoder. This cell must never mention recon_embed.

log_dir = Path(train_cfg["log_dir"])
ckpt_dir = Path(train_cfg["checkpoint_dir"])
results_dir = Path(train_cfg["results_dir"])
for p in (log_dir, ckpt_dir, results_dir):
    p.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(log_dir))

steps = int(STEPS_OVERRIDE) if STEPS_OVERRIDE is not None else int(train_cfg["steps"])
batch_size = int(train_cfg["batch_size"])
seq_len = int(train_cfg["seq_len"])
log_every = int(train_cfg["log_every"])
image_every = int(train_cfg["image_every"])
ckpt_every = int(train_cfg["checkpoint_every"])
context_len = max(1, seq_len // 4)

history: list[dict] = []
last_recon_vis = None
last_video_vis = None
steps_per_sec = None
last_log_time = time.time()
last_log_step = start_step


def rolling_mean(values: list[float], window: int = 5) -> list[float]:
    out = []
    for i in range(len(values)):
        lo = max(0, i - window + 1)
        out.append(sum(values[lo : i + 1]) / (i - lo + 1))
    return out


def make_video_pred_image(context_recon, imagined_recon, real_obs_u8):
    """Truth / model (context+imagined) / error for one sequence."""
    truth = real_obs_u8[0].float() / 255.0
    model_frames = torch.cat([context_recon[0], imagined_recon[0]], dim=0)
    model_frames = model_frames.permute(0, 2, 3, 1).clamp(0.0, 1.0).cpu()
    error = (model_frames - truth + 1.0) / 2.0

    def row(frames: torch.Tensor) -> np.ndarray:
        imgs = [(f.clamp(0.0, 1.0) * 255.0).round().numpy().astype(np.uint8) for f in frames]
        return np.concatenate(imgs, axis=1)

    return np.concatenate([row(truth), row(model_frames), row(error)], axis=0)


def show_progress(history, recon_vis, video_vis, steps_per_sec=None):
    clear_output(wait=False)
    xs = [h["step"] for h in history]
    fig = plt.figure(figsize=(13, 8))

    ax0 = fig.add_subplot(2, 2, 1)
    for key, color in [("recon_l1", "#264653"), ("reward", "#e76f51"), ("continue", "#2a9d8f")]:
        raw = [h[key] for h in history]
        ax0.plot(xs, raw, color=color, alpha=0.25, linewidth=1)
        ax0.plot(xs, rolling_mean(raw), label=key, color=color, linewidth=1.6)
    ax0.set_yscale("log")
    ax0.set_title("loss terms (faint=raw, bold=rolling mean, log y)")
    ax0.legend(fontsize=8)

    ax1 = fig.add_subplot(2, 2, 2)
    dyn_ys = [h["kl_dyn_raw"] for h in history]
    rep_ys = [h["kl_rep_raw"] for h in history]
    ax1.plot(xs, dyn_ys, label="kl_dyn_raw", color="#e76f51", ls="--", linewidth=2.0)
    ax1.plot(xs, rep_ys, label="kl_rep_raw", color="#2a9d8f", ls="-", linewidth=1.4, alpha=0.9)
    free_nats = float(train_cfg["free_nats"])
    ax1.axhline(free_nats, color="gray", ls=":", linewidth=1.5, label=f"free_nats={free_nats:g}")
    data_max = max(dyn_ys + rep_ys + [0.0])
    ax1.set_ylim(0.0, max(free_nats * 1.25, min(data_max * 1.15, free_nats * 6.0), 1.0))
    ax1.set_title("KL raw (total, summed over stoch groups)")
    ax1.legend(fontsize=8)

    if recon_vis is not None:
        real, pred = recon_vis
        ax2 = fig.add_subplot(2, 2, 3)
        ax2.imshow(np.concatenate([real[0].numpy(), pred[0].numpy()], axis=1))
        ax2.set_title("real | posterior recon (mid-sequence)", fontsize=9)
        ax2.axis("off")

    if video_vis is not None:
        ax3 = fig.add_subplot(2, 2, 4)
        ax3.imshow(video_vis)
        ax3.set_title(f"open-loop (context={context_len}): truth / model / error", fontsize=8)
        ax3.axis("off")

    fig.tight_layout()
    display(fig)
    plt.close(fig)
    plt.close("all")

    h = history[-1]
    sps = f" ({steps_per_sec:.2f} steps/s)" if steps_per_sec else ""
    vram = vram_peak_gb()
    vram_s = f"  vram {vram[0]:.1f}/{vram[1]:.1f} GiB" if vram else ""
    print(
        f"step {h['step']:5d}/{steps}  total={h['total']:.4f}  "
        f"recon={h['recon']:.2f} recon_l1={h['recon_l1']:.4f}  "
        f"rew={h['reward']:.4f} rew_mae={h['reward_mae']:.4f} cont={h['continue']:.4f}  "
        f"kl={h['kl']:.4f} (dyn_raw={h['kl_dyn_raw']:.3f} rep_raw={h['kl_rep_raw']:.3f})"
        f"{sps}{vram_s}",
        flush=True,
    )


model.train()
print(f"training to step {steps} on {device} (start={start_step})...", flush=True)
for step in range(start_step + 1, steps + 1):
    batch = buffer.sample(batch_size, seq_len)
    _loss, metrics = world_model_step(
        model,
        optim,
        batch,
        device=device,
        train_cfg=train_cfg,
        amp_dtype=amp_dtype,
        scaler=scaler,
    )
    metrics["step"] = step

    if step % log_every == 0 or step == start_step + 1:
        for k, v in metrics.items():
            if k != "step":
                writer.add_scalar(f"m3/{k}", v, step)
        history.append(metrics)
        now = time.time()
        steps_per_sec = (step - last_log_step) / max(now - last_log_time, 1e-6)
        last_log_time, last_log_step = now, step
        if step % image_every != 0 and step != start_step + 1:
            show_progress(history, last_recon_vis, last_video_vis, steps_per_sec)

    if step % image_every == 0 or step == start_step + 1:
        model.eval()
        with torch.no_grad(), autocast_context(device, amp_dtype):
            vis = buffer.sample(min(4, batch_size), seq_len)
            vis_g = to_device(vis, device)
            v_out = model(vis_g["obs"], vis_g["actions"])
            t_vis = seq_len // 2
            real = vis["obs"][:, t_vis].cpu()
            pred = nchw_unit_to_nhwc_uint8(v_out.recon[:, t_vis].detach().cpu())
            last_recon_vis = (real, pred)
            from PIL import Image

            strips = [
                np.concatenate([real[i].numpy(), pred[i].numpy()], axis=1)
                for i in range(real.shape[0])
            ]
            Image.fromarray(np.concatenate(strips, axis=0), mode="RGB").save(
                results_dir / f"recon_step_{step:05d}.png"
            )

            vp = model.video_predict(vis_g["obs"], vis_g["actions"], context_len=context_len)
            video_img = make_video_pred_image(vp.context_recon, vp.imagined_recon, vis["obs"])
            last_video_vis = video_img
            Image.fromarray(video_img, mode="RGB").save(
                results_dir / f"video_pred_step_{step:05d}.png"
            )
        model.train()
        if history:
            show_progress(
                history,
                last_recon_vis,
                last_video_vis,
                steps_per_sec if step == last_log_step else None,
            )

    if step % ckpt_every == 0:
        path = ckpt_dir / f"ckpt_step_{step:05d}.pt"
        torch.save(
            {"step": step, "model": model.state_dict(), "optim": optim.state_dict()},
            path,
        )
        print(f"wrote {path}", flush=True)

final = ckpt_dir / "ckpt_final.pt"
torch.save({"step": steps, "model": model.state_dict(), "optim": optim.state_dict()}, final)
(results_dir / "train_metrics.json").write_text(json.dumps(history, indent=2))
writer.flush()
writer.close()
show_progress(history, last_recon_vis, last_video_vis)

model.eval()
with torch.no_grad(), autocast_context(device, amp_dtype):
    vis = buffer.sample(8, seq_len)
    vis_g = to_device(vis, device)
    v_out = model(vis_g["obs"], vis_g["actions"])
    real = vis["obs"][:, -1].cpu()
    pred = nchw_unit_to_nhwc_uint8(v_out.recon[:, -1].detach().cpu())
    from PIL import Image

    strips = [
        np.concatenate([real[i].numpy(), pred[i].numpy()], axis=1)
        for i in range(real.shape[0])
    ]
    Image.fromarray(np.concatenate(strips, axis=0), mode="RGB").save(
        results_dir / "recon_final.png"
    )
    vp = model.video_predict(vis_g["obs"], vis_g["actions"], context_len=context_len)
    video_img = make_video_pred_image(vp.context_recon, vp.imagined_recon, vis["obs"])
    Image.fromarray(video_img, mode="RGB").save(results_dir / "video_pred_final.png")
model.train()
print("done", final)


training to step 50000 on cuda (start=0)...


TypeError: Got unsupported ScalarType BFloat16

## Exit criteria — did this run actually work?

M3's bar: all four losses trending down, KL neither dead nor exploded, reward
prediction correlating with real reward, open-loop rollouts that do not
collapse to a constant frame.

Look at `recon_final.png` and `video_pred_final.png`. Blurry but recognizable
Crafter is a pass. Readable inventory digits are not the bar.


In [ ]:
assert len(history) >= 2, "need at least two logged points to judge a trend"

first, last = history[0], history[-1]
checks: list[tuple[str, bool, str]] = []

for key, min_drop in [("recon_l1", 0.3), ("reward", 0.1), ("continue", 0.3)]:
    a, b = first[key], last[key]
    drop = 1.0 - (b / max(a, 1e-8))
    checks.append(
        (
            f"{key} loss dropped >= {min_drop:.0%} (first={a:.4f} -> last={b:.4f}, actual={drop:.0%})",
            drop >= min_drop,
            f"{key} isn't improving -- check lr/scale or a graph bug",
        )
    )

kl_rep_last = last["kl_rep_raw"]
checks.append(
    (
        f"kl_rep_raw is not dead (last={kl_rep_last:.3f} > 0.02)",
        kl_rep_last > 0.02,
        "posterior has collapsed onto the prior -- latent carries ~no information",
    )
)
checks.append(
    (
        f"kl_rep_raw has not exploded (last={kl_rep_last:.3f} < 30.0)",
        kl_rep_last < 30.0,
        "posterior is ignoring the prior entirely -- imagination rollouts will drift badly",
    )
)

model.eval()
with torch.no_grad(), autocast_context(device, amp_dtype):
    vis = buffer.sample(8, seq_len)
    vis_g = to_device(vis, device)
    v_out = model(vis_g["obs"], vis_g["actions"])
    obs_std = float(nhwc_uint8_to_nchw_unit(vis_g["obs"]).std())
    recon_std = float(v_out.recon.float().std())
    vp = model.video_predict(vis_g["obs"], vis_g["actions"], context_len=context_len)
    imagined_std = float(vp.imagined_recon.float().std())
model.train()

ratio = recon_std / max(obs_std, 1e-8)
checks.append(
    (
        f"recon pixel std is close to real (recon_std={recon_std:.4f} vs obs_std={obs_std:.4f}, ratio={ratio:.2f})",
        ratio > 0.4,
        "recon has collapsed toward a constant (solid-color) output",
    )
)
ratio_img = imagined_std / max(obs_std, 1e-8)
checks.append(
    (
        f"open-loop imagined recon std is not collapsed (imagined_std={imagined_std:.4f} vs obs_std={obs_std:.4f}, ratio={ratio_img:.2f})",
        ratio_img > 0.15,
        "prior rollouts collapse to a constant frame -- inspect video_pred_final.png",
    )
)

true_chunks, pred_chunks = [], []
with torch.no_grad(), autocast_context(device, amp_dtype):
    for _ in range(20):
        rb = buffer.sample(batch_size, seq_len)
        rb_g = to_device(rb, device)
        r_out = model(rb_g["obs"], rb_g["actions"])
        true_chunks.append(rb["rewards"].numpy().reshape(-1))
        decoded = symlog_twohot_mean(r_out.reward_pred, model.reward_head.bins)
        pred_chunks.append(decoded.float().cpu().numpy().reshape(-1))
reward_true = np.concatenate(true_chunks)
reward_pred = np.concatenate(pred_chunks)
if reward_true.std() > 1e-8 and reward_pred.std() > 1e-8:
    reward_corr = float(np.corrcoef(reward_true, reward_pred)[0, 1])
    checks.append(
        (
            f"reward prediction correlates with real reward (r={reward_corr:.2f})",
            reward_corr > 0.3,
            "reward head isn't tracking real reward -- latent may not be carrying reward-relevant information",
        )
    )
else:
    print("[SKIP] reward correlation: too few nonzero-reward examples in this sample")

print(f"steps trained: {last['step']}\n")
all_pass = True
for description, ok, hint in checks:
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {description}")
    if not ok:
        print(f"       -> {hint}")
        all_pass = False

print()
print("RESULT:", "PASS — M3 world model looks healthy" if all_pass else "FAIL — see hints above")
